# needle-fp4: paper experiments

Runs the four studies of the paper on a Colab GPU and saves everything to Drive under
`MyDrive/10714/fp4-needle/results/paper/`. Runtime > Change runtime type > **GPU**, then Run all.

| Study | Question | Data |
|---|---|---|
| E1 storage | Which 4-bit format stores real weights with the least error per bit; does the crest factor predict it; does rotation help INT4 but hurt NVFP4? | Our trained MLP, LSTM and GPT; GPT-2 (124M) published weights |
| E2 threshold | Does FP4 training stall when the gradient-to-noise ratio falls below sqrt(3), and does switching the backward pass to FP32 there close the gap? | Mini-GPT on text8 |
| E3 data size | Is the FP4 round-to-nearest regularization effect on the LSTM real, and does it shrink with more data? | Penn Treebank, 2000 lines to the full 42k |
| E4 model size | Does the FP4 gap grow with model size? | Mini-GPT (0.1M, 0.8M, 4.7M parameters) on text8 |

`QUICK = True` checks that everything runs (about 20 to 40 minutes). `QUICK = False` is the full run (many hours;
each finished run is saved to Drive immediately). At the end, the last cell zips `results/paper` so you can share it.

*AI-assisted: written with Claude (Anthropic) via Claude Code; reviewed and tested by the team.*

In [ ]:
BRANCH = "main"
DEVICE = "cuda"
QUICK = True
USE_GPT2 = True          # E1 also analyses GPT-2's published weights (548 MB download, read with NumPy)
SEEDS = [0] if QUICK else [0, 1, 2]
GPT_STEPS = 200 if QUICK else 3000
GPT_EVAL = 100 if QUICK else 250
GPT_SIZES = ["tiny"] if QUICK else ["tiny", "small", "medium"]
TEXT8_CHARS = 2_000_000 if QUICK else 20_000_000
LSTM_LINES = [2000] if QUICK else [2000, 10000, 0]      # 0 = all of PTB
LSTM_EPOCHS = 1 if QUICK else 3
R = "results/paper"

In [ ]:
import base64, subprocess
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/10714

# Private repo: create a GitHub token with read access to this repo, then in Colab click the key
# icon (Secrets) > Add new secret, name it GITHUB_TOKEN and allow notebook access. The token is
# passed to git as a header for each command and is never written to Drive.
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
basic = base64.b64encode(f"x-access-token:{token}".encode()).decode() if token else ""
AUTH = f"-c http.extraheader='Authorization: Basic {basic}'" if token else ""
print("Using a GitHub token from Colab Secrets" if token else "No GITHUB_TOKEN secret: cloning as public")

!test -d fp4-needle || git {AUTH} clone https://github.com/shreya-v7/fp4-needle.git
%cd /content/drive/MyDrive/10714/fp4-needle
# Match GitHub exactly. Colab autosaves into the Drive copy of this notebook, which would
# otherwise make `git pull` refuse to update and leave old code in place.
# (Only tracked files are reset; results/, data/ and build/ are git-ignored and kept.)
!git {AUTH} fetch origin && git checkout -f {BRANCH} && git reset --hard origin/{BRANCH}
import subprocess
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
remote = subprocess.check_output(['git', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
assert head == remote, f"Checkout is at {head[:7]}, but origin/{BRANCH} is {remote[:7]}; see the git output above."
!git log --oneline -3

In [ ]:
!pip3 install -q pybind11
!make clean && make
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
sys.path.append('python'); sys.path.append('apps')
import needle as ndl
print('CUDA backend built:', ndl.cuda().enabled())
if DEVICE == 'cuda' and not ndl.cuda().enabled():
    raise RuntimeError("No CUDA backend: Runtime > Change runtime type > GPU, then Runtime > Run all again.")

In [ ]:
!python3 -m pytest tests/fp4 -q

## Data: MNIST, Penn Treebank, text8, GPT-2 weights

In [ ]:
import os, hashlib, shutil, urllib.request

# MNIST files and the MD5 of the copy our homeworks used (identical to the course repo).
MNIST = {"train-images-idx3-ubyte.gz": "0cb5796cd8e7c7bffef03edbdb01fac5",
         "train-labels-idx1-ubyte.gz": "23e0cdb926d142c2d470a3641a19d1ff",
         "t10k-images-idx3-ubyte.gz":  "3d4db374c734e4f92d346e063bf6e41a",
         "t10k-labels-idx1-ubyte.gz":  "0407fd84edb9a00bbad73c377ff95de9"}
os.makedirs("data", exist_ok=True)
for name, md5 in MNIST.items():
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        # Prefer a homework copy on Drive; otherwise download from the course's hw0 repo.
        local = [p for p in (f"../{hw}/data/{name}" for hw in ("hw2", "hw1", "hw0")) if os.path.exists(p)]
        if local:
            shutil.copy(local[0], dst)
        else:
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/dlsyscourse/hw0/main/data/{name}", dst)
    assert hashlib.md5(open(dst, "rb").read()).hexdigest() == md5, f"{dst} is corrupted; delete it and rerun"
    print("ok", name)

In [ ]:
import os, urllib.request, zipfile
os.makedirs('data/ptb', exist_ok=True)
for split in ['train', 'test', 'valid']:
    path = f'data/ptb/{split}.txt'
    if not os.path.exists(path):
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.{split}.txt', path)
if not os.path.exists('data/text8'):
    urllib.request.urlretrieve('http://mattmahoney.net/dc/text8.zip', 'data/text8.zip')
    zipfile.ZipFile('data/text8.zip').extractall('data')
if USE_GPT2 and not os.path.exists('data/gpt2/model.safetensors'):
    os.makedirs('data/gpt2', exist_ok=True)
    urllib.request.urlretrieve('https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors',
                               'data/gpt2/model.safetensors')
print({p: os.path.getsize(p) for p in ['data/ptb/train.txt', 'data/text8'] + (['data/gpt2/model.safetensors'] if USE_GPT2 else [])})

## E1: storage (post-training quantization of real weights)
Train one FP32 model of each kind, then quantize its weights with every 4-bit format, with and without a
Hadamard rotation, and measure (a) weight error vs bits and crest factor, (b) test accuracy / perplexity /
bits per character with the quantized weights.

In [ ]:
!python3 apps/fp4_experiment.py --configs grid --only fp32 --device {DEVICE} --epochs 3 --optimizer adam --lr 0.001 --out {R}/e1_mlp.csv --save-weights "{R}/ckpt/mlp_{{config}}_s{{seed}}.npz"
!python3 apps/fp4_lstm_experiment.py --configs grid --only fp32 --device {DEVICE} --max-lines {LSTM_LINES[-1]} --epochs {LSTM_EPOCHS} --out {R}/e1_lstm.csv --save-weights "{R}/ckpt/lstm_{{config}}_s{{seed}}.npz"
!python3 apps/fp4_gpt_experiment.py --configs main --only fp32 --device {DEVICE} --data-file data/text8 --max-chars {TEXT8_CHARS} --size small --steps {GPT_STEPS} --eval-every {GPT_EVAL} --out {R}/e1_gpt.csv --save-weights "{R}/ckpt/gpt_{{config}}_{{size}}_s{{seed}}.npz"
GPT2 = '--gpt2 data/gpt2/model.safetensors' if USE_GPT2 else ''
!python3 apps/fp4_storage.py --checkpoints {R}/ckpt/mlp_fp32_s0.npz {R}/ckpt/lstm_fp32_s0.npz {R}/ckpt/gpt_fp32_small_s0.npz {GPT2} --ptq --device {DEVICE} --out-dir {R}/storage
!python3 apps/plot_paper.py ptq {R}/storage/ptq.csv --out {R}/storage/ptq.md
from IPython.display import Image, Markdown, display
display(Image(f'{R}/storage/storage.png'))
display(Markdown(open(f'{R}/storage/storage_summary.md').read()))
display(Markdown(open(f'{R}/storage/ptq.md').read()))

## E2: the sqrt(3) gradient-to-noise threshold
Train the mini-GPT fully in FP4 (round-to-nearest and stochastic rounding) while logging
r = ||g|| / ||g_q - g|| every few steps, then repeat with the backward pass switched to FP32 at two points.
Prediction (Chmiel et al., 2025): progress stalls once r < sqrt(3), and switching after that closes the gap.

In [ ]:
MON = 10 if QUICK else 50
for seed in SEEDS:
    !python3 apps/fp4_gpt_experiment.py --configs main --device {DEVICE} --data-file data/text8 --max-chars {TEXT8_CHARS} --size small --steps {GPT_STEPS} --eval-every {GPT_EVAL} --seed {seed} --monitor-every {MON} --out {R}/e2_gpt_seed{seed}.csv
    for sw in [GPT_STEPS // 4, GPT_STEPS // 2]:
        !python3 apps/fp4_gpt_experiment.py --configs main --only fprop+dgrad+wgrad+sr --device {DEVICE} --data-file data/text8 --max-chars {TEXT8_CHARS} --size small --steps {GPT_STEPS} --eval-every {GPT_EVAL} --seed {seed} --monitor-every {MON} --switch-step {sw} --out {R}/e2_gpt_switch{sw}_seed{seed}.csv
!python3 apps/plot_paper.py threshold {R}/e2_gpt_*seed0*.csv --out {R}/fig_threshold.png
display(Image(f'{R}/fig_threshold.png'))

## E3: LSTM data size (is the regularization effect real?)
FP32, all-FP4 round-to-nearest and all-FP4 + SR on growing slices of Penn Treebank.

In [ ]:
for lines in LSTM_LINES:
    for seed in SEEDS:
        !python3 apps/fp4_lstm_experiment.py --configs grid --only fp32 fprop+dgrad+wgrad fprop+dgrad+wgrad+sr --device {DEVICE} --max-lines {lines} --epochs {LSTM_EPOCHS} --seed {seed} --out {R}/e3_lstm_L{lines}_seed{seed}.csv
!python3 apps/plot_paper.py datasize {R}/e3_lstm_L*_seed*.csv --out {R}/fig_datasize.png
display(Image(f'{R}/fig_datasize.png'))

## E4: model size (does the FP4 gap grow?)

In [ ]:
for size in GPT_SIZES:
    for seed in SEEDS:
        !python3 apps/fp4_gpt_experiment.py --configs main --device {DEVICE} --data-file data/text8 --max-chars {TEXT8_CHARS} --size {size} --steps {GPT_STEPS} --eval-every {GPT_EVAL} --seed {seed} --out {R}/e4_gpt_{size}_seed{seed}.csv
!python3 apps/plot_paper.py scaling {R}/e4_gpt_*_seed*.csv --out {R}/fig_scaling.png
display(Image(f'{R}/fig_scaling.png'))

## Share the results
Zips `results/paper` (CSVs, plots, tables; checkpoints excluded). In Drive, share `paper_results.zip`
with "Anyone with the link" (or download it and attach it), and send the link so the results can be analysed.

In [ ]:
import shutil, os
shutil.make_archive('paper_results', 'zip', 'results', 'paper')
os.system('zip -q -d paper_results.zip "paper/ckpt/*"')
print('Created', os.path.abspath('paper_results.zip'), os.path.getsize('paper_results.zip') // 1024, 'KB')